In [63]:
# Cell 1: imports, configuration, reproducibility, directories

import os
import json
import random
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

SEED = 42
DATA_PATH = "dataset/EVSE-B-PowerCombined_filtered.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE_DIR = Path("./gridsearch_runs")
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
RESULTS_DIR = BASE_DIR / "results"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

NUMERIC_COLS = ["shunt_voltage", "bus_voltage_V", "current_mA", "power_mW"]
CATEGORICAL_COLS = ["State"]
TARGET_COL = "Attack"

EXPECTED_STATE_VALUES = {"idle", "charging"}
EXPECTED_ATTACK_VALUES = {"syn-flood", "none", "Backdoor"}

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

print("Device:", DEVICE)
print("Data path exists:", os.path.exists(DATA_PATH))

Device: cuda
Data path exists: True


In [64]:
# Cell 2: load dataset and keep only the required columns

df = pd.read_csv(DATA_PATH)

#required_cols = NUMERIC_COLS + CATEGORICAL_COLS
required_cols = NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df.loc[:, required_cols].copy()

for col in NUMERIC_COLS:
    if not pd.api.types.is_numeric_dtype(df[col]):
        raise TypeError(f"Column {col} must be numeric.")

for col in CATEGORICAL_COLS:
    if df[col].isna().any():
        raise ValueError(f"Column {col} contains missing values.")

print("Dataset shape after column selection:", df.shape)
display(df.head())

Dataset shape after column selection: (49017, 6)


,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack
0,978,5.165,1027,5300,idle,syn-flood
1,872,5.161,1009,4980,idle,syn-flood
2,1017,5.165,1029,5300,idle,syn-flood
3,930,5.161,1005,5180,idle,syn-flood
4,958,5.165,1034,5180,idle,syn-flood


In [65]:
# Cell 3: validate categorical values and basic assumptions

check_cols = NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL]

if df[check_cols].isnull().any().any():
    raise ValueError("Dataset contains missing values, but none were expected.")

if df[NUMERIC_COLS + CATEGORICAL_COLS + [TARGET_COL]].isnull().any().any():
    raise ValueError("Dataset contains missing values, but none were expected.")

for col in CATEGORICAL_COLS:
    if df[col].isnull().any():
        raise ValueError(f"Column {col} contains missing values.")

if df[TARGET_COL].isnull().any():
    raise ValueError(f"Target column {TARGET_COL} contains missing values.")

state_values = set(df["State"].astype(str).str.strip().str.lower().unique())
attack_values = set(df[TARGET_COL].astype(str).str.strip().unique())

print("Observed State values:", state_values)
print("Observed Attack values:", attack_values)

unexpected_state = state_values - {v.lower() for v in EXPECTED_STATE_VALUES}
unexpected_attack = attack_values - EXPECTED_ATTACK_VALUES

if unexpected_state:
    raise ValueError(f"Unexpected State values found: {unexpected_state}")

if unexpected_attack:
    raise ValueError(f"Unexpected Attack values found: {unexpected_attack}")

Observed State values: {'idle', 'charging'}
Observed Attack values: {'Backdoor', 'syn-flood', 'none'}


In [66]:
# Cell 4: optional cleanup for consistent category spelling

df_original = df.copy(deep=True)

df["State"] = df["State"].astype(str).str.strip()
df["Attack"] = df["Attack"].astype(str).str.strip()

display(df.sample(min(5, len(df)), random_state=SEED))

,shunt_voltage,bus_voltage_V,current_mA,power_mW,State,Attack
20330,559,5.189,694,3640,charging,none
19931,671,5.197,513,2620,charging,none
44521,623,5.185,621,3380,idle,Backdoor
25919,481,5.201,454,2320,idle,none
30086,667,5.173,614,2940,charging,Backdoor


In [67]:
# Cell 5: fixed train/validation/test split
# Stratify by Attack so the 3 label classes are preserved across splits.

X_train_full, X_test = train_test_split(
    df,
    test_size=0.15,
    random_state=SEED,
    shuffle=True,
    stratify=df["Attack"]
)

X_train, X_val = train_test_split(
    X_train_full,
    test_size=0.1765,  # Gives about 70/15/15 overall
    random_state=SEED,
    shuffle=True,
    stratify=X_train_full["Attack"]
)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

print("\nAttack distribution in train:")
print(X_train["Attack"].value_counts(normalize=True))

print("\nAttack distribution in val:")
print(X_val["Attack"].value_counts(normalize=True))

print("\nAttack distribution in test:")
print(X_test["Attack"].value_counts(normalize=True))
print(df["Attack"].value_counts())

Train: (34310, 6)
Val: (7354, 6)
Test: (7353, 6)

Attack distribution in train:
Backdoor     0.431215
none         0.293034
syn-flood    0.275751
Name: Attack, dtype: float64

Attack distribution in val:
Backdoor     0.431194
none         0.293038
syn-flood    0.275768
Name: Attack, dtype: float64

Attack distribution in test:
Backdoor     0.431253
none         0.292942
syn-flood    0.275806
Name: Attack, dtype: float64
Backdoor     21137
none         14363
syn-flood    13517
Name: Attack, dtype: int64


In [68]:
# Cell 6: fit preprocessors on train split only
# Numeric features are standardized; categorical features are one-hot encoded.

scaler = StandardScaler()

try:
    ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
except TypeError:
    ohe = OneHotEncoder(sparse=False, handle_unknown="ignore")

X_train_num = scaler.fit_transform(X_train[NUMERIC_COLS])
X_val_num = scaler.transform(X_val[NUMERIC_COLS])
X_test_num = scaler.transform(X_test[NUMERIC_COLS])

X_train_cat = ohe.fit_transform(X_train[CATEGORICAL_COLS])
X_val_cat = ohe.transform(X_val[CATEGORICAL_COLS])
X_test_cat = ohe.transform(X_test[CATEGORICAL_COLS])

try:
    cat_feature_names = ohe.get_feature_names_out(CATEGORICAL_COLS).tolist()
except AttributeError:
    cat_feature_names = ohe.get_feature_names(CATEGORICAL_COLS).tolist()

X_train_processed = np.hstack([X_train_num, X_train_cat]).astype(np.float32)
X_val_processed = np.hstack([X_val_num, X_val_cat]).astype(np.float32)
X_test_processed = np.hstack([X_test_num, X_test_cat]).astype(np.float32)

FEATURE_NAMES = NUMERIC_COLS + cat_feature_names
INPUT_DIM = X_train_processed.shape[1]

print("Encoded feature names:", FEATURE_NAMES)
print("Input dimension:", INPUT_DIM)

Encoded feature names: ['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW', 'State_charging', 'State_idle']
Input dimension: 6


In [69]:
# Cell 7: convert to tensors and build denoising loaders with class labels

from sklearn.preprocessing import LabelEncoder

X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_processed, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_processed, dtype=torch.float32)

label_encoder = LabelEncoder()
label_encoder.fit(df[TARGET_COL].astype(str))

y_train = label_encoder.transform(X_train[TARGET_COL].astype(str))
y_val = label_encoder.transform(X_val[TARGET_COL].astype(str))
y_test = label_encoder.transform(X_test[TARGET_COL].astype(str))

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

def add_gaussian_noise(x, noise_factor=0.1):
    noise = torch.randn_like(x) * noise_factor
    return x + noise

def make_loaders(batch_size, noise_factor=0.1, denoise_val=True, denoise_test=False):
    X_train_noisy = add_gaussian_noise(X_train_tensor, noise_factor=noise_factor)

    if denoise_val:
        X_val_noisy = add_gaussian_noise(X_val_tensor, noise_factor=noise_factor)
    else:
        X_val_noisy = X_val_tensor.clone()

    if denoise_test:
        X_test_noisy = add_gaussian_noise(X_test_tensor, noise_factor=noise_factor)
    else:
        X_test_noisy = X_test_tensor.clone()

    train_ds = TensorDataset(X_train_noisy, X_train_tensor, y_train_tensor)
    val_ds = TensorDataset(X_val_noisy, X_val_tensor, y_val_tensor)
    test_ds = TensorDataset(X_test_noisy, X_test_tensor, y_test_tensor)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

In [70]:
# Cell 8: denoising autoencoder with classifier head for 3-class classification

class DenoisingAutoencoderClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=3, latent_dim=16, hidden_dims=(128, 64), dropout=0.0):
        super().__init__()

        encoder_layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            encoder_layers.extend([
                nn.Linear(prev_dim, h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = h
        encoder_layers.append(nn.Linear(prev_dim, latent_dim))
        self.encoder = nn.Sequential(*encoder_layers)

        decoder_layers = []
        prev_dim = latent_dim
        for h in reversed(hidden_dims):
            decoder_layers.extend([
                nn.Linear(prev_dim, h),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = h
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)

        self.classifier = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(latent_dim, num_classes)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        logits = self.classifier(z)
        return x_hat, logits

In [71]:
# Cell 9: training, validation, checkpointing, and final metrics logging

from sklearn.metrics import f1_score

def evaluate_loss(model, loader, criterion_recon, criterion_cls, alpha=1.0, beta=1.0):
    model.eval()
    losses = []

    with torch.no_grad():
        for xb, x_clean, yb in loader:
            xb = xb.to(DEVICE)
            x_clean = x_clean.to(DEVICE)
            yb = yb.to(DEVICE)

            x_hat, logits = model(xb)
            loss_recon = criterion_recon(x_hat, x_clean)
            loss_cls = criterion_cls(logits, yb)
            loss = alpha * loss_recon + beta * loss_cls
            losses.append(loss.item())

    return float(np.mean(losses))

def evaluate_reconstruction_error(model, loader):
    model.eval()
    errors = []

    with torch.no_grad():
        for xb, x_clean, yb in loader:
            xb = xb.to(DEVICE)
            x_clean = x_clean.to(DEVICE)

            x_hat, _ = model(xb)
            batch_error = torch.mean((x_hat - x_clean) ** 2, dim=1)
            errors.extend(batch_error.detach().cpu().numpy().tolist())

    return float(np.mean(errors))

def evaluate_macro_f1(model, loader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for xb, x_clean, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            _, logits = model(xb)
            pred = torch.argmax(logits, dim=1)

            y_true.extend(yb.detach().cpu().numpy().tolist())
            y_pred.extend(pred.detach().cpu().numpy().tolist())

    return float(f1_score(y_true, y_pred, average="macro"))

def save_checkpoint(model, optimizer, epoch, best_val_loss, params, path):
    ckpt = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_loss": best_val_loss,
        "params": params
    }
    torch.save(ckpt, path)

def append_result_row(result, results_path):
    if results_path.exists():
        df_old = pd.read_csv(results_path)
        df_new = pd.concat([df_old, pd.DataFrame([result])], ignore_index=True)
    else:
        df_new = pd.DataFrame([result])

    df_new = df_new.sort_values("best_val_loss", ascending=True)
    df_new.to_csv(results_path, index=False)

def train_autoencoder(params, max_epochs=80, patience=5):
    set_seed(SEED)

    train_loader, val_loader, test_loader = make_loaders(
        batch_size=params["batch_size"],
        noise_factor=params["noise_factor"],
        denoise_val=params["denoise_val"],
        denoise_test=params["denoise_test"]
    )

    model = DenoisingAutoencoderClassifier(
        input_dim=INPUT_DIM,
        num_classes=3,
        latent_dim=params["latent_dim"],
        hidden_dims=params["hidden_dims"],
        dropout=params["dropout"]
    ).to(DEVICE)

    criterion_recon = nn.MSELoss()
    criterion_cls = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=params["learning_rate"],
        weight_decay=params["weight_decay"]
    )

    run_name = (
        f"daeclf_ld{params['latent_dim']}"
        f"_hd{'-'.join(map(str, params['hidden_dims']))}"
        f"_bs{params['batch_size']}"
        f"_lr{params['learning_rate']}"
        f"_do{params['dropout']}"
        f"_wd{params['weight_decay']}"
        f"_nf{params['noise_factor']}"
        f"_vnoise{int(params['denoise_val'])}"
        f"_tnoise{int(params['denoise_test'])}"
    )

    checkpoint_path = CHECKPOINT_DIR / f"{run_name}.pt"

    best_val_loss = float("inf")
    best_epoch = -1
    wait = 0

    alpha = params.get("alpha", 1.0)
    beta = params.get("beta", 1.0)

    for epoch in range(1, max_epochs + 1):
        model.train()
        train_losses = []

        for xb, x_clean, yb in train_loader:
            xb = xb.to(DEVICE)
            x_clean = x_clean.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            x_hat, logits = model(xb)
            loss_recon = criterion_recon(x_hat, x_clean)
            loss_cls = criterion_cls(logits, yb)
            loss = alpha * loss_recon + beta * loss_cls
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        val_loss = evaluate_loss(model, val_loader, criterion_recon, criterion_cls, alpha=alpha, beta=beta)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            wait = 0
            save_checkpoint(model, optimizer, epoch, best_val_loss, params, checkpoint_path)
        else:
            wait += 1

        if wait >= patience:
            break

    best_ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(best_ckpt["model_state_dict"])

    test_loss = evaluate_loss(model, test_loader, criterion_recon, criterion_cls, alpha=alpha, beta=beta)
    reconstruction_error = evaluate_reconstruction_error(model, test_loader)
    macro_f1 = evaluate_macro_f1(model, test_loader)

    result = {
        "run_name": run_name,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "test_loss": test_loss,
        "reconstruction_error": reconstruction_error,
        "macro_f1": macro_f1,
        "alpha": alpha,
        "beta": beta,
        "input_dim": INPUT_DIM,
        **params
    }

    return result

In [72]:
# Cell 10: smaller stage 1 coarse grid

param_grid_stage1 = {
    "latent_dim": [4, 8, 16],            # Broad check of small vs larger bottleneck.
    "hidden_dims": [(16,), (16, 8)],  # Compare shallow vs slightly deeper network.
    "dropout": [0.0, 0.1],            # Light regularization check.
    "batch_size": [128],              # Keep fixed in stage 1 to reduce runs.
    "learning_rate": [1e-3, 1e-4],    # Broad optimizer step-size comparison.
    "weight_decay": [0.0],            # Keep fixed in stage 1.
    "noise_factor": [0.05],           # Keep one moderate denoising level first.
    "denoise_val": [True],            # Validation stays noisy for DAE tuning.
    "denoise_test": [False],          # Test stays clean for final evaluation.
    "alpha": [0.5, 1.0],              # Compare lower vs equal reconstruction weight.
    "beta": [1.0]                     # Keep classification weight fixed.
}

grid_keys = list(param_grid_stage1.keys())
grid_values = list(param_grid_stage1.values())

all_combinations_stage1 = [
    dict(zip(grid_keys, values))
    for values in product(*grid_values)
]

print("Number of stage 1 runs:", len(all_combinations_stage1))
display(pd.DataFrame(all_combinations_stage1).head(10))

Number of stage 1 runs: 48


,latent_dim,hidden_dims,dropout,batch_size,learning_rate,weight_decay,noise_factor,denoise_val,denoise_test,alpha,beta
0,4,"(16,)",0.0,128,0.0010,0.0,0.05,True,False,0.5,1.0
1,4,"(16,)",0.0,128,0.0010,0.0,0.05,True,False,1.0,1.0
2,4,"(16,)",0.0,128,0.0001,0.0,0.05,True,False,0.5,1.0
3,4,"(16,)",0.0,128,0.0001,0.0,0.05,True,False,1.0,1.0
4,4,"(16,)",0.1,128,0.0010,0.0,0.05,True,False,0.5,1.0
5,4,"(16,)",0.1,128,0.0010,0.0,0.05,True,False,1.0,1.0
6,4,"(16,)",0.1,128,0.0001,0.0,0.05,True,False,0.5,1.0
7,4,"(16,)",0.1,128,0.0001,0.0,0.05,True,False,1.0,1.0
8,4,"(16, 8)",0.0,128,0.0010,0.0,0.05,True,False,0.5,1.0
9,4,"(16, 8)",0.0,128,0.0010,0.0,0.05,True,False,1.0,1.0


In [73]:
# Cell 11: stage 1 coarse search

results_path_stage1 = RESULTS_DIR / "dae_stage1_results.csv"
dae_results_stage1 = []

if results_path_stage1.exists():
    results_path_stage1.unlink()

for i, params in enumerate(all_combinations_stage1, start=1):
    print(f"Running stage 1 {i}/{len(all_combinations_stage1)}: {params}")
    result = train_autoencoder(params, max_epochs=80, patience=5)
    dae_results_stage1.append(result)
    append_result_row(result, results_path_stage1)
    print(f"Updated results file: {results_path_stage1}")

dae_results_stage1_df = pd.read_csv(results_path_stage1).sort_values("best_val_loss", ascending=True)
display(dae_results_stage1_df.head(10))

Running stage 1 1/48: {'latent_dim': 4, 'hidden_dims': (16,), 'dropout': 0.0, 'batch_size': 128, 'learning_rate': 0.001, 'weight_decay': 0.0, 'noise_factor': 0.05, 'denoise_val': True, 'denoise_test': False, 'alpha': 0.5, 'beta': 1.0}
Updated results file: gridsearch_runs/results/dae_stage1_results.csv
Running stage 1 2/48: {'latent_dim': 4, 'hidden_dims': (16,), 'dropout': 0.0, 'batch_size': 128, 'learning_rate': 0.001, 'weight_decay': 0.0, 'noise_factor': 0.05, 'denoise_val': True, 'denoise_test': False, 'alpha': 1.0, 'beta': 1.0}
Updated results file: gridsearch_runs/results/dae_stage1_results.csv
Running stage 1 3/48: {'latent_dim': 4, 'hidden_dims': (16,), 'dropout': 0.0, 'batch_size': 128, 'learning_rate': 0.0001, 'weight_decay': 0.0, 'noise_factor': 0.05, 'denoise_val': True, 'denoise_test': False, 'alpha': 0.5, 'beta': 1.0}
Updated results file: gridsearch_runs/results/dae_stage1_results.csv
Running stage 1 4/48: {'latent_dim': 4, 'hidden_dims': (16,), 'dropout': 0.0, 'batch_si

,run_name,best_epoch,best_val_loss,test_loss,reconstruction_error,macro_f1,alpha,beta,input_dim,latent_dim,hidden_dims,dropout,batch_size,learning_rate,weight_decay,noise_factor,denoise_val,denoise_test
0,daeclf_ld16_hd16_bs128_lr0.001_do0.0_wd0.0_nf0...,80,0.385351,0.380594,0.000880,0.827796,1.0,1.0,6,16,"(16,)",0.0,128,0.001,0.0,0.05,True,False
1,daeclf_ld8_hd16-8_bs128_lr0.001_do0.0_wd0.0_nf...,76,0.386796,0.379935,0.004319,0.821281,1.0,1.0,6,8,"(16, 8)",0.0,128,0.001,0.0,0.05,True,False
2,daeclf_ld16_hd16_bs128_lr0.001_do0.0_wd0.0_nf0...,47,0.390442,0.384577,0.001230,0.820024,0.5,1.0,6,16,"(16,)",0.0,128,0.001,0.0,0.05,True,False
3,daeclf_ld16_hd16-8_bs128_lr0.001_do0.0_wd0.0_n...,76,0.391514,0.379550,0.003351,0.827469,1.0,1.0,6,16,"(16, 8)",0.0,128,0.001,0.0,0.05,True,False
4,daeclf_ld16_hd16-8_bs128_lr0.001_do0.0_wd0.0_n...,75,0.391775,0.389040,0.028931,0.826010,0.5,1.0,6,16,"(16, 8)",0.0,128,0.001,0.0,0.05,True,False
5,daeclf_ld8_hd16-8_bs128_lr0.001_do0.0_wd0.0_nf...,68,0.392469,0.385925,0.003259,0.818699,0.5,1.0,6,8,"(16, 8)",0.0,128,0.001,0.0,0.05,True,False
6,daeclf_ld8_hd16_bs128_lr0.001_do0.0_wd0.0_nf0....,63,0.405166,0.402404,0.000768,0.811960,0.5,1.0,6,8,"(16,)",0.0,128,0.001,0.0,0.05,True,False
7,daeclf_ld8_hd16_bs128_lr0.001_do0.0_wd0.0_nf0....,79,0.407108,0.398048,0.000752,0.812495,1.0,1.0,6,8,"(16,)",0.0,128,0.001,0.0,0.05,True,False
8,daeclf_ld16_hd16_bs128_lr0.001_do0.1_wd0.0_nf0...,43,0.414439,0.416404,0.015414,0.808027,0.5,1.0,6,16,"(16,)",0.1,128,0.001,0.0,0.05,True,False
9,daeclf_ld4_hd16-8_bs128_lr0.001_do0.0_wd0.0_nf...,78,0.422734,0.419189,0.010437,0.812101,1.0,1.0,6,4,"(16, 8)",0.0,128,0.001,0.0,0.05,True,False


In [74]:
# Cell 12: inspect the best stage 1 configuration

best_dae_stage1 = dae_results_stage1_df.iloc[0].to_dict()
print("Best stage 1 DAE configuration:")
print(json.dumps(best_dae_stage1, indent=2, default=str))

Best stage 1 DAE configuration:
{
  "run_name": "daeclf_ld16_hd16_bs128_lr0.001_do0.0_wd0.0_nf0.05_vnoise1_tnoise0",
  "best_epoch": 80,
  "best_val_loss": 0.3853510510304878,
  "test_loss": 0.3805935023159816,
  "reconstruction_error": 0.0008803296124616,
  "macro_f1": 0.8277960588401282,
  "alpha": 1.0,
  "beta": 1.0,
  "input_dim": 6,
  "latent_dim": 16,
  "hidden_dims": "(16,)",
  "dropout": 0.0,
  "batch_size": 128,
  "learning_rate": 0.001,
  "weight_decay": 0.0,
  "noise_factor": 0.05,
  "denoise_val": true,
  "denoise_test": false
}


In [75]:
# Cell 13: save metadata for reproducibility

split_info = {
    "seed": SEED,
    "data_path": DATA_PATH,
    "numeric_columns": NUMERIC_COLS,
    "categorical_columns": CATEGORICAL_COLS,
    "encoded_feature_names": FEATURE_NAMES,
    "input_dim": INPUT_DIM,
    "train_shape": list(X_train.shape),
    "val_shape": list(X_val.shape),
    "test_shape": list(X_test.shape),
    "state_values": sorted(df["State"].unique().tolist()),
    "attack_values": sorted(df["Attack"].unique().tolist()),
    "model_type": "denoising_autoencoder_classifier",
    "search_strategy": "coarse_to_fine_stage1"
}

with open(RESULTS_DIR / "split_metadata.json", "w") as f:
    json.dump(split_info, f, indent=2)

print("Saved:")
print("-", RESULTS_DIR / "dae_stage1_results.csv")
print("-", RESULTS_DIR / "split_metadata.json")
print("-", CHECKPOINT_DIR)

Saved:
- gridsearch_runs/results/dae_stage1_results.csv
- gridsearch_runs/results/split_metadata.json
- gridsearch_runs/checkpoints


In [76]:
# Cell 14: build stage 2 fine grid from best stage 1 result

best_latent = int(best_dae_stage1["latent_dim"])
best_dropout = float(best_dae_stage1["dropout"])
best_bs = int(best_dae_stage1["batch_size"])
best_lr = float(best_dae_stage1["learning_rate"])
best_noise = float(best_dae_stage1["noise_factor"])
best_alpha = float(best_dae_stage1["alpha"])
best_hidden = best_dae_stage1["hidden_dims"]

stage2_latent = sorted({max(2, best_latent - 2), best_latent, best_latent + 2})
stage2_dropout = sorted({max(0.0, best_dropout - 0.05), best_dropout, min(0.3, best_dropout + 0.05)})
stage2_batch_size = sorted({max(32, best_bs // 2), best_bs, min(512, best_bs * 2)})
stage2_learning_rate = sorted({best_lr / 2, best_lr, best_lr * 2})
stage2_noise_factor = sorted({max(0.01, best_noise - 0.02), best_noise, best_noise + 0.02})
stage2_alpha = sorted({max(0.1, best_alpha - 0.25), best_alpha, best_alpha + 0.25})

if best_hidden == (16,):
    stage2_hidden_dims = [(16,), (32,), (32, 16)]
else:
    stage2_hidden_dims = [(16, 8), (32, 16), (32, 16, 8)]

param_grid_stage2 = {
    "latent_dim": stage2_latent,
    "hidden_dims": stage2_hidden_dims,
    "dropout": stage2_dropout,
    "batch_size": stage2_batch_size,
    "learning_rate": stage2_learning_rate,
    "weight_decay": [0.0],
    "noise_factor": stage2_noise_factor,
    "denoise_val": [True],
    "denoise_test": [False],
    "alpha": stage2_alpha,
    "beta": [1.0]
}

grid_keys = list(param_grid_stage2.keys())
grid_values = list(param_grid_stage2.values())

all_combinations_stage2 = [
    dict(zip(grid_keys, values))
    for values in product(*grid_values)
]

print("Number of stage 2 runs:", len(all_combinations_stage2))
display(pd.DataFrame(all_combinations_stage2).head(10))

Number of stage 2 runs: 1458


,latent_dim,hidden_dims,dropout,batch_size,learning_rate,weight_decay,noise_factor,denoise_val,denoise_test,alpha,beta
0,14,"(16, 8)",0.0,64,0.0005,0.0,0.03,True,False,0.75,1.0
1,14,"(16, 8)",0.0,64,0.0005,0.0,0.03,True,False,1.00,1.0
2,14,"(16, 8)",0.0,64,0.0005,0.0,0.03,True,False,1.25,1.0
3,14,"(16, 8)",0.0,64,0.0005,0.0,0.05,True,False,0.75,1.0
4,14,"(16, 8)",0.0,64,0.0005,0.0,0.05,True,False,1.00,1.0
5,14,"(16, 8)",0.0,64,0.0005,0.0,0.05,True,False,1.25,1.0
6,14,"(16, 8)",0.0,64,0.0005,0.0,0.07,True,False,0.75,1.0
7,14,"(16, 8)",0.0,64,0.0005,0.0,0.07,True,False,1.00,1.0
8,14,"(16, 8)",0.0,64,0.0005,0.0,0.07,True,False,1.25,1.0
9,14,"(16, 8)",0.0,64,0.0010,0.0,0.03,True,False,0.75,1.0


In [ ]:
# Cell 15: stage 2 fine search

results_path_stage2 = RESULTS_DIR / "dae_stage2_results.csv"
dae_results_stage2 = []

if results_path_stage2.exists():
    results_path_stage2.unlink()

for i, params in enumerate(all_combinations_stage2, start=1):
    print(f"Running stage 2 {i}/{len(all_combinations_stage2)}: {params}")
    result = train_autoencoder(params, max_epochs=100, patience=5)
    dae_results_stage2.append(result)
    append_result_row(result, results_path_stage2)
    print(f"Updated results file: {results_path_stage2}")

dae_results_stage2_df = pd.read_csv(results_path_stage2).sort_values("best_val_loss", ascending=True)
display(dae_results_stage2_df.head(10))

Running stage 2 1/1458: {'latent_dim': 14, 'hidden_dims': (16, 8), 'dropout': 0.0, 'batch_size': 64, 'learning_rate': 0.0005, 'weight_decay': 0.0, 'noise_factor': 0.030000000000000002, 'denoise_val': True, 'denoise_test': False, 'alpha': 0.75, 'beta': 1.0}
Updated results file: gridsearch_runs/results/dae_stage2_results.csv
Running stage 2 2/1458: {'latent_dim': 14, 'hidden_dims': (16, 8), 'dropout': 0.0, 'batch_size': 64, 'learning_rate': 0.0005, 'weight_decay': 0.0, 'noise_factor': 0.030000000000000002, 'denoise_val': True, 'denoise_test': False, 'alpha': 1.0, 'beta': 1.0}
Updated results file: gridsearch_runs/results/dae_stage2_results.csv
Running stage 2 3/1458: {'latent_dim': 14, 'hidden_dims': (16, 8), 'dropout': 0.0, 'batch_size': 64, 'learning_rate': 0.0005, 'weight_decay': 0.0, 'noise_factor': 0.030000000000000002, 'denoise_val': True, 'denoise_test': False, 'alpha': 1.25, 'beta': 1.0}
Updated results file: gridsearch_runs/results/dae_stage2_results.csv
Running stage 2 4/1458:

In [ ]:
# Cell 16: inspect best stage 2 configuration

best_dae_stage2 = dae_results_stage2_df.iloc[0].to_dict()
print("Best stage 2 DAE configuration:")
print(json.dumps(best_dae_stage2, indent=2, default=str))